# End-to-End Data Science Pipeline - Practice

**Goal:** Demonstrate the full DS pipeline: Data Loading → EDA → Cleaning → Feature Engineering → Modeling → Evaluation

**Dataset:** Titanic - Predict survival based on passenger attributes

## 1. Setup & Data Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_style('whitegrid')
%matplotlib inline

/home/vahid/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [ ]:
# Load data
df = pd.read_csv('Dataset/train.csv')

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

df['Survived'].value_counts().plot.bar(ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Survival Count')
axes[0].set_xticklabels(['Died', 'Survived'], rotation=0)

df['Survived'].value_counts().plot.pie(ax=axes[1], autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
axes[1].set_ylabel('')
axes[1].set_title('Survival Rate')

plt.tight_layout()
plt.show()

print(f"Survival rate: {df['Survived'].mean():.2%}")

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing': missing, 'Percent': missing_pct}).query('Missing > 0').sort_values('Percent', ascending=False)

In [ ]:
# Survival by key features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.barplot(x='Sex', y='Survived', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Survival by Sex')

sns.barplot(x='Pclass', y='Survived', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Survival by Class')

sns.histplot(df, x='Age', hue='Survived', bins=30, ax=axes[1, 0], kde=True)
axes[1, 0].set_title('Survival by Age')

sns.barplot(x='Embarked', y='Survived', data=df, ax=axes[1, 1])
axes[1, 1].set_title('Survival by Embarkation Port')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlations')
plt.tight_layout()
plt.show()

### EDA Key Findings
- **Sex:** Women survived at ~74% vs men at ~19%
- **Pclass:** 1st class passengers had highest survival rate (~63%)
- **Age:** Children had higher survival rates
- **Missing data:** Cabin (77%), Age (20%), Embarked (<1%)

## 3. Data Cleaning & Preprocessing

In [ ]:
# Work on a copy
data = df.copy()

# Fill missing Age with median by Pclass + Sex (more accurate than global median)
data['Age'] = data.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Fill missing Embarked with mode
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

# Drop Cabin (too many missing values) and Ticket (not useful)
data = data.drop(columns=['Cabin', 'Ticket', 'PassengerId'])

print(f"Remaining missing values: {data.isnull().sum().sum()}")
data.head()

## 4. Feature Engineering

In [ ]:
# Extract title from Name
data['Title'] = data['Name'].str.extract(r',\s*([^.]+)\.').squeeze()

# Group rare titles
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs', 'Don': 'Rare',
    'Dona': 'Rare', 'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'the Countess': 'Rare'
}
data['Title'] = data['Title'].map(title_map).fillna('Rare')

# Family size
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1

# Is alone
data['IsAlone'] = (data['FamilySize'] == 1).astype(int)

# Age bins
data['AgeBin'] = pd.cut(data['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Child', 'Teen', 'Young', 'Mid', 'Senior'])

# Fare bins
data['FareBin'] = pd.qcut(data['Fare'], q=4, labels=['Low', 'Medium', 'High', 'VeryHigh'])

# Drop Name (replaced by Title)
data = data.drop(columns=['Name'])

data.head()

In [ ]:
# Encode categorical variables
data_encoded = pd.get_dummies(data, columns=['Sex', 'Embarked', 'Title', 'AgeBin', 'FareBin'], drop_first=True)

print(f"Final shape: {data_encoded.shape}")
data_encoded.head()

## 5. Modeling

In [ ]:
# Split features and target
X = data_encoded.drop(columns=['Survived'])
y = data_encoded['Survived']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train survival rate: {y_train.mean():.2%}, Test survival rate: {y_test.mean():.2%}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    # Use scaled data for Logistic Regression, raw for tree-based
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, scaler.transform(X), y, cv=5, scoring='accuracy')
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'accuracy': acc, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(), 'y_pred': y_pred}
    print(f"{name:25s} | Test Acc: {acc:.4f} | CV Acc: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 6. Model Evaluation

In [ ]:
# Pick the best model for detailed evaluation
best_name = max(results, key=lambda k: results[k]['cv_mean'])
best_pred = results[best_name]['y_pred']

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=['Died', 'Survived']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=['Died', 'Survived']).plot(ax=ax, cmap='Blues')
ax.set_title(f'Confusion Matrix - {best_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (for tree-based models)
if best_name in ['Random Forest', 'Gradient Boosting']:
    best_model = models[best_name]
    importance = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
    
    plt.figure(figsize=(8, 6))
    importance.tail(15).plot.barh()
    plt.title(f'Top 15 Feature Importances - {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
# Model comparison chart
comparison = pd.DataFrame({name: {'Test Accuracy': r['accuracy'], 'CV Accuracy': r['cv_mean']} 
                           for name, r in results.items()}).T

comparison.plot.bar(figsize=(8, 4), rot=0)
plt.title('Model Comparison')
plt.ylabel('Accuracy')
plt.ylim(0.7, 0.9)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning (Bonus - if time permits)

In [ ]:
# GridSearch on the best tree-based model
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.4f}")
print(f"Test score: {grid.score(X_test, y_test):.4f}")

## Summary

| Step | What was done |
|------|---------------|
| Data Loading | Loaded CSV, inspected shape, types, and statistics |
| EDA | Analyzed target distribution, survival rates by features, correlations |
| Cleaning | Imputed Age (median by group), Embarked (mode), dropped Cabin/Ticket |
| Feature Engineering | Extracted Title, created FamilySize, IsAlone, age/fare bins |
| Modeling | Compared Logistic Regression, Random Forest, Gradient Boosting |
| Evaluation | Classification report, confusion matrix, feature importance, cross-validation |